# Snap Media Browser for Colab

This notebook clones the repo, mounts Google Drive, creates an isolated virtualenv with a compatible Gradio build, prints diagnostics, and launches `snap_media_browser.py` against a Drive folder.

Run the cells from top to bottom. The last cell keeps running while the app is live.

The notebook avoids mutating Colab's global Python environment, so runtime debugging stays reproducible.

In [ ]:
#@title 1. Clone the repo and create an isolated runtime
REPO_URL = "https://github.com/ivyking48/Gradio-ClaudeCode-Tester.git"  #@param {type:"string"}
BRANCH = "codex-fix-snap-media-browser-docs"  #@param {type:"string"}
VENV_DIR = "/content/snap-media-browser-venv"  #@param {type:"string"}

%cd /content
!rm -rf Gradio-ClaudeCode-Tester "$VENV_DIR"
!git clone --branch "$BRANCH" "$REPO_URL"
!python3 -m venv "$VENV_DIR"
!"$VENV_DIR/bin/python" -m pip install -q --upgrade pip setuptools wheel
!"$VENV_DIR/bin/python" -m pip install -q "gradio>=6,<7" "Pillow>=11.0,<12"
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
%cd /content/Gradio-ClaudeCode-Tester

In [ ]:
#@title 2. Mount Drive and choose app settings
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

MEDIA_ROOT = "/content/drive/MyDrive/SnapInsta_Downloads"  #@param {type:"string"}
THUMB_DIR = "/content/_snap_thumbs"  #@param {type:"string"}
SHARE_PUBLIC = True  #@param {type:"boolean"}
DEBUG_LOGS = True  #@param {type:"boolean"}

os.environ["SNAP_MEDIA_ROOT"] = MEDIA_ROOT
os.environ["SNAP_THUMB_DIR"] = THUMB_DIR
os.environ["SNAP_SHARE"] = "1" if SHARE_PUBLIC else "0"
os.environ["SNAP_SKIP_COLAB_MOUNT"] = "1"
os.environ["SNAP_DEBUG"] = "1" if DEBUG_LOGS else "0"

print("SNAP_MEDIA_ROOT=", os.environ["SNAP_MEDIA_ROOT"])
print("SNAP_THUMB_DIR=", os.environ["SNAP_THUMB_DIR"])
print("SNAP_SHARE=", os.environ["SNAP_SHARE"])
print("SNAP_SKIP_COLAB_MOUNT=", os.environ["SNAP_SKIP_COLAB_MOUNT"])
print("SNAP_DEBUG=", os.environ["SNAP_DEBUG"])

In [ ]:
# 3. Preflight diagnostics
import os
import pathlib
import subprocess

repo = pathlib.Path('/content/Gradio-ClaudeCode-Tester')
venv_python = pathlib.Path('/content/snap-media-browser-venv/bin/python')

print('repo exists:', repo.exists())
print('venv python exists:', venv_python.exists())
print('media root exists:', pathlib.Path(os.environ['SNAP_MEDIA_ROOT']).exists())
print('thumb dir exists:', pathlib.Path(os.environ['SNAP_THUMB_DIR']).exists())
if pathlib.Path(os.environ['SNAP_MEDIA_ROOT']).exists():
    try:
        print('media item count:', len(list(pathlib.Path(os.environ['SNAP_MEDIA_ROOT']).iterdir())))
    except Exception as exc:
        print('media item count failed:', exc)

code = r'''
import inspect, os, sys
import gradio as gr
from PIL import Image
print('python:', sys.version)
print('executable:', sys.executable)
print('gradio:', getattr(gr, '__version__', 'unknown'))
print('pillow:', getattr(Image, '__version__', 'unknown'))
print('html_signature:', inspect.signature(gr.HTML.__init__))
print('html_has_custom_api:', all(name in inspect.signature(gr.HTML.__init__).parameters for name in ['html_template', 'css_template', 'js_on_load']))
print('env_media_root:', os.environ.get('SNAP_MEDIA_ROOT'))
print('env_thumb_dir:', os.environ.get('SNAP_THUMB_DIR'))
print('env_share:', os.environ.get('SNAP_SHARE'))
print('env_skip_mount:', os.environ.get('SNAP_SKIP_COLAB_MOUNT'))
print('env_debug:', os.environ.get('SNAP_DEBUG'))
'''
subprocess.run([str(venv_python), '-c', code], check=True)

In [ ]:
# 4. Launch the app
%cd /content/Gradio-ClaudeCode-Tester
!PYTHONUNBUFFERED=1 /content/snap-media-browser-venv/bin/python snap_media_browser.py